In [15]:
import torch
import torch.nn as nn
import numpy as np
from collections import Counter

In [44]:
with open("lstm_training_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()
    
words = text.split()
print(text[:200])

word_counts = Counter(words)

vocab = sorted(word_counts)

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

vocab_size = len(vocab)

encoded = np.array([word2idx[w] for w in words])

print("Vocabulary size:", vocab_size)

datasets are split into training validation and testing sets deep
learning uses neural networks with many layers large datasets help
neural networks generalize better large datasets help neural networ
Vocabulary size: 85


In [45]:
seq_length = 20

inputs = []
targets = []

for i in range(len(encoded) - seq_length):
    
    inputs.append(encoded[i:i+seq_length])
    targets.append(encoded[i+1:i+seq_length+1])

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

print(inputs.shape)

torch.Size([4980, 20])


In [46]:
split = int(0.8 * len(inputs))

train_x = inputs[:split]
val_x = inputs[split:]

train_y = targets[:split]
val_y = targets[split:]

In [47]:
class WordLSTM(nn.Module):
    
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(WordLSTM, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        
        x = self.embedding(x)
        
        outputs, hidden = self.lstm(x, hidden)
        
        outputs = self.fc(outputs)
        
        return outputs, hidden

In [48]:
embed_size = 64
hidden_size = 128
num_layers = 2

model = WordLSTM(vocab_size, embed_size, hidden_size, num_layers)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [55]:
epochs = 50
batch_size = 32

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    # Shuffle training data
    perm = torch.randperm(train_x.size(0))
    train_x = train_x[perm]
    train_y = train_y[perm]

    for i in range(0, len(train_x), batch_size):

        x = train_x[i:i+batch_size]
        y = train_y[i:i+batch_size]

        optimizer.zero_grad()

        output, _ = model(x)

        loss = criterion(
            output.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(output, dim=2)

        correct += (preds == y).sum().item()
        total += y.numel()

    avg_loss = total_loss / (len(train_x)//batch_size)

    train_accuracy = correct / total

    train_perplexity = np.exp(avg_loss)

    # ===============================
    # Validation (INSIDE EPOCH LOOP)
    # ===============================

    model.eval()

    with torch.no_grad():

        val_output, _ = model(val_x)

        val_loss = criterion(
            val_output.reshape(-1, vocab_size),
            val_y.reshape(-1)
        )

        val_perplexity = torch.exp(val_loss)

        preds = torch.argmax(val_output, dim=2)
        val_accuracy = (preds == val_y).sum().item() / val_y.numel()

    print(f"\nEpoch {epoch+1}")

    print("Train Loss:", avg_loss)
    print("Train Perplexity:", train_perplexity)
    print("Train Accuracy:", train_accuracy)

    print("Validation Loss:", val_loss.item())
    print("Validation Perplexity:", val_perplexity.item())
    print("Validation Accuracy:", val_accuracy)


Epoch 1
Train Loss: 0.24259426468803036
Train Perplexity: 1.2745513886899047
Train Accuracy: 0.9082454819277108
Validation Loss: 0.6371315121650696
Validation Perplexity: 1.8910486698150635
Validation Accuracy: 0.8639558232931727

Epoch 2
Train Loss: 0.23764908854519168
Train Perplexity: 1.2682640663408866
Train Accuracy: 0.9108308232931727
Validation Loss: 0.6418560743331909
Validation Perplexity: 1.9000041484832764
Validation Accuracy: 0.863855421686747

Epoch 3
Train Loss: 0.235721543791794
Train Perplexity: 1.26582178515321
Train Accuracy: 0.9104166666666667
Validation Loss: 0.6426719427108765
Validation Perplexity: 1.9015549421310425
Validation Accuracy: 0.8640562248995984

Epoch 4
Train Loss: 0.2319262611769861
Train Perplexity: 1.2610267387723768
Train Accuracy: 0.9112951807228916
Validation Loss: 0.6623613834381104
Validation Perplexity: 1.9393664598464966
Validation Accuracy: 0.8625

Epoch 5
Train Loss: 0.23097829388514643
Train Perplexity: 1.2598318930966732
Train Accuracy: 

In [56]:
def generate_text(model, start_text, length=20):

    model.eval()

    words = start_text.lower().split()

    input_seq = torch.tensor([word2idx[w] for w in words]).unsqueeze(0)

    hidden = None

    generated = words.copy()

    for _ in range(length):

        output, hidden = model(input_seq, hidden)

        probs = torch.softmax(output[:, -1, :], dim=1)

        word_idx = torch.multinomial(probs, 1).item()

        word = idx2word[word_idx]

        generated.append(word)

        input_seq = torch.tensor([[word_idx]])

    return " ".join(generated)

In [57]:
print(generate_text(model, "datasets are ", 40))

datasets are split into training validation and testing sets feature engineering can improve predictive performance optimization methods like adam and gradient descent train models deep learning uses neural networks with many layers lstm networks are useful for sequence prediction tasks pytorch provides
